## Build review queue (blind double-review)

In [ ]:
import pandas as pd

In [ ]:
REVIEWERS = ["reviewer2@orcid", "reviewer3@orcid"]

def assign_reviewers(df: pd.DataFrame, reviewers: list[str], k: int = 2) -> pd.DataFrame:
    """Deterministic round-robin assignment of k reviewers per row."""
    df = df.copy()
    queue = df[df["triage"].isin(["review", "spot_check"])].reset_index(drop=True)
    n = len(reviewers)
    assignments = []
    for i in range(len(queue)):
        picks = [reviewers[(i + j) % n] for j in range(k)]
        assignments.append(" | ".join(picks))
    queue["assigned_reviewers"] = assignments
    queue["reviewer1_decision"] = ""
    queue["reviewer2_decision"] = ""
    queue["reviewer_notes"] = ""
    # Reorder columns so reviewers see context first, LLM justification, but NOT confidence
    display_cols = ["subject_id","subject_label","object_id","object_label","object_source",
                    "predicate","justification","syntactic_compatibility","transformation_note",
                    "assigned_reviewers","reviewer1_decision","reviewer2_decision","reviewer_notes"]
    return queue[display_cols + [c for c in queue.columns if c not in display_cols]]

review_queue = assign_reviewers(pairs_df, REVIEWERS, k=2)
review_queue.to_csv(OUT_DIR / "review_queue.tsv", sep="\t", index=False)
print(f"Queued {len(review_queue)} pairs for review across {len(REVIEWERS)} reviewers")

## Ingest reviewer decisions and compute agreement

In [ ]:
from sklearn.metrics import cohen_kappa_score

def ingest_reviews(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path, sep="\t", dtype=str).fillna("")
    df["agreed"] = df["reviewer1_decision"] == df["reviewer2_decision"]
    df["final_predicate"] = np.where(
        df["agreed"],
        df["reviewer1_decision"],
        "NEEDS_DISCUSSION"
    )
    # kappa (drop empty)
    both = df[(df["reviewer1_decision"] != "") & (df["reviewer2_decision"] != "")]
    if len(both) > 5:
        kappa = cohen_kappa_score(both["reviewer1_decision"], both["reviewer2_decision"])
        print(f"Cohen's kappa (reviewers 1 vs 2): {kappa:.3f}  (n={len(both)})")
    return df

# reviewed = ingest_reviews(OUT_DIR / "review_queue_completed.tsv")

## Emit SSSOM TSV

SSSOM_HEADER = f"""# curie_map:
#   faire: https://example.org/faire/
#   skos: http://www.w3.org/2004/02/skos/core#
#   semapv: https://w3id.org/semapv/vocab/
#   dwc: http://rs.tdwg.org/dwc/terms/
#   mixs: https://w3id.org/mixs/
#   nmdc: https://w3id.org/nmdc/
#   obi: http://purl.obolibrary.org/obo/OBI_
#   envo: http://purl.obolibrary.org/obo/ENVO_
#   uberon: http://purl.obolibrary.org/obo/UBERON_
# mapping_set_id: https://example.org/faire/mappings/v1
# mapping_set_version: 0.1.0
# mapping_set_description: FAIReSheets v1.2 mappings to DwC, MIxS, NMDC, MIQE, Humboldt, INSDC
# license: https://creativecommons.org/licenses/by/4.0/
# mapping_provider: FAIReSheets working group
# mapping_tool: hybrid-retrieval + MapperGPT-style LLM adjudication + double-blind human review
# mapping_date: {date.today().isoformat()}
# extension_definitions:
#   - slot_name: syntactic_compatibility